# Warehouse Analysis Notebook

This notebook loads the flattened Parquet tables from the Data Warehouse and displays them.

**Tables:** `genes`, `gene_seeds`, `powers`, `power_seeds`, `gene_regulation`, `gene_side_effects`.

In [1]:
import os
import sys
from pyspark.sql import SparkSession
import pandas as pd

# Ensure we can import our modules
# sys.path.append(os.path.abspath("../super-services/src")) # Adjust if needed

from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe

In [2]:
# Initialize Spark Session
bootstrap_spark_env()

spark = (SparkSession.builder
.appName("WarehouseViewer")
.config("spark.executor.memory", "4g")
.config("spark.driver.memory", "4g")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 02:35:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/19 02:35:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/19 02:35:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
# Load Configuration
conf = utils.get_app_conf("generate_powers")
stage_root = conf.get_string("stage_root")
warehouse_root = os.path.join(stage_root, "warehouse")

print(f"Reading Warehouse from: {warehouse_root}")

Reading Warehouse from: /home/gideon/tmp/super_powers/data/warehouse


## 1. Genes Table

In [4]:
genes_df = spark.read.parquet(os.path.join(warehouse_root, "genes"))
display_scrollable_dataframe(genes_df.toPandas())

## 2. Powers Table

In [5]:
powers_df = spark.read.parquet(os.path.join(warehouse_root, "powers"))
display_scrollable_dataframe(powers_df.toPandas())

## 3. Link Tables
**Gene Seeds, Power Seeds, Side Effects**

In [6]:
gene_seeds_df = spark.read.parquet(os.path.join(warehouse_root, "gene_seeds"))
display_scrollable_dataframe(gene_seeds_df.limit(100).toPandas())

,gene_id,seed_name,role,weight
0,ELASTI-569,Elastic Physiology,primary,0.70
1,ELASTI-569,Rubber Physiology,primary,0.20
2,HAMMER-783,Hammerspace Access,primary,0.70
3,HAMMER-783,Probability Manipulation,primary,0.20
4,COSMIC-745,Cosmic Empowerment,primary,0.70
5,COSMIC-745,Regeneration,primary,0.15
6,COSMIC-745,Narrative Awareness,primary,0.15
7,REGENE-986,Regeneration,primary,0.60
8,REGENE-986,Mythic Invulnerability,primary,0.40
9,REGENE-494,Regeneration,primary,0.75


In [7]:
power_seeds_df = spark.read.parquet(os.path.join(warehouse_root, "power_seeds"))
display_scrollable_dataframe(power_seeds_df.limit(100).toPandas())

,power_id,seed_name,role,weight
0,1a4b0ac2f22b,Concept Suppression,secondary,0.2
1,c6034ac92dec,Super Breath,secondary,0.1
2,d5ed082fef54,Concept Suppression,secondary,0.1
3,6c403d9c9dab,Concept Suppression,secondary,0.3
4,c2572da732d2,Super Breath,secondary,0.2
5,9e5010af116a,Concept Suppression,secondary,0.2
6,68a63c565684,Super Breath,secondary,0.3
7,e60dae64583d,Concept Suppression,secondary,0.1
8,07a5bcc58d68,Super Breath,secondary,0.1
9,699f787cd581,Concept Suppression,secondary,0.2


In [8]:
side_effects_df = spark.read.parquet(os.path.join(warehouse_root, "gene_side_effects"))
display_scrollable_dataframe(side_effects_df.limit(100).toPandas())

,gene_id,side_effect_name,probability,severity,trigger_condition
0,HAMMER-321,Humiliation Trigger,0.40,4,Occurs when attempting to retrieve an object larger than the user's current mass.
1,HAMMER-321,Cynical Deconstruction,0.35,3,Activates if the object retrieved is referenced ironically.
2,HAMMER-321,Moral Whiplash,0.50,5,Triggers when objects of conflicting moral alignment are retrieved consecutively.
3,HAMMER-321,Genre Drift,0.40,3,Triggered if the object retrieved is thematically inconsistent with the environment.
4,OPTICE-847,Moral Whiplash,0.50,3,Activation of energy emission without focus.
5,OPTICE-847,Monologue Compulsion,0.40,4,Upon consecutive emission bursts.
6,OPTICE-847,Rule Awareness,0.30,2,Extended use beyond one hour.
7,OPTICE-847,Escalation Lock,0.60,5,When energy emission encounters force resistance.
8,OPTICE-847,Tragic Irony,0.40,3,Failure to activate force field during critical moments.
9,ELASTI-658,Monologue Compulsion,0.35,3,During extended physical deformation.


## 4. Gene Regulation (Graph)

In [9]:
reg_df = spark.read.parquet(os.path.join(warehouse_root, "gene_regulation"))
display_scrollable_dataframe(reg_df.limit(100).toPandas())

,source_gene_id,target_gene_id,effect,strength
0,PAINTE-642,PAINTE-101,amplify,0.77
1,PAINTE-642,PAINTE-239,destabilize,0.53
2,PAINTE-642,PAINTE-340,gate,0.42
3,PAINTE-642,PAINTE-445,suppress,0.38
4,PAINTE-642,PAINTE-556,amplify,0.49
5,PAINTE-642,PAINTE-662,suppress,0.61
6,PAINTE-642,PAINTE-774,destabilize,0.65
7,PAINTE-642,PAINTE-885,gate,0.57
8,PAINTE-642,PAINTE-920,amplify,0.71
9,PAINTE-642,PAINTE-301,suppress,0.35


## 5. Sanity Export (Excel)
Exporting top 10 rows of each table to `data/sanity/warehouse_top10.xlsx`.

In [10]:
sanity_dir = os.path.join(stage_root, "sanity")
os.makedirs(sanity_dir, exist_ok=True)
out_path = os.path.join(sanity_dir, "warehouse_top10.xlsx")

# Forced Deletion to ensure freshness
if os.path.exists(out_path):
    try:
        os.remove(out_path)
        print(f"Deleted existing file: {out_path}")
    except OSError as e:
        print(f"Warning: Could not delete existing file {out_path}: {e}")

print(f"Exporting to {out_path}...")

with pd.ExcelWriter(out_path, mode='w') as writer:
    genes_df.limit(10).toPandas().to_excel(writer, sheet_name="Genes", index=False)
    powers_df.limit(10).toPandas().to_excel(writer, sheet_name="Powers", index=False)
    gene_seeds_df.limit(10).toPandas().to_excel(writer, sheet_name="Gene Seeds", index=False)
    power_seeds_df.limit(10).toPandas().to_excel(writer, sheet_name="Power Seeds", index=False)
    side_effects_df.limit(10).toPandas().to_excel(writer, sheet_name="Gene Side Effects", index=False)
    reg_df.limit(10).toPandas().to_excel(writer, sheet_name="Regulation", index=False)

print("Export Complete. File is fresh.")

Exporting to /home/gideon/tmp/super_powers/data/sanity/warehouse_top10.xlsx...
Export Complete. File is fresh.
